In [1]:
import pandas as pd
from collections import defaultdict
import json
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa


In [2]:
# Load JSON file
def input_data(annot):
    with open("giorgos.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # If the JSON is already a list of records
    if isinstance(data, list):
        df = pd.DataFrame(data)
    
    # If the JSON is a dict with a key containing the records
    elif isinstance(data, dict):
        # Change 'items' to the correct top-level key if needed
        if "items" in data:
            df = pd.DataFrame(data["items"])
        else:
            # Fallback: normalize nested dict structure
            df = pd.json_normalize(data)
    return df

In [3]:
annot1 = input_data("giorgos.json") 
annot2 = input_data("maria.json") 
annot3 = input_data("angeliki.json") 
annot4 = input_data("konstantinos.json") 

### Ground truth

In [4]:
def ground_truth(df):
    real_attack = defaultdict(list)
    contains_attack = defaultdict(list)
    gold_attack = defaultdict(list)
    for v in df.data.values:
        real_attack[v['task_uid']]=v['attack_type']
        contains_attack[v['task_uid']]=v['contains_attack_gold']
        gold_attack[v['task_uid']]=v['gold_label']
    return real_attack, contains_attack, gold_attack

In [5]:
real_1, contains_1, gold_1 = ground_truth(annot1)
real_2, contains_2, gold_2 = ground_truth(annot2)
real_3, contains_3, gold_3 = ground_truth(annot3)
real_4, contains_4, gold_4 = ground_truth(annot4)

### Annotated

In [6]:
def annotations(df):
    annotated_attack = defaultdict(list)
    for i, v in zip(df.annotations, df.data.values):
        annotated_attack[v['task_uid']]=i[0]['result'][0]['value']['choices'][0]
    return annotated_attack

In [7]:
annotated_1 = annotations(annot1)
annotated_2 = annotations(annot2)
annotated_3 = annotations(annot3)
annotated_4 = annotations(annot4)

### Dict to matrix

In [8]:
def dicts_to_annotation_matrix(annotator_dicts, missing_value=np.nan):
    """
    Convert a list of annotator dicts into a 2D numpy array.

    Parameters
    ----------
    annotator_dicts : list of dict
        One dict per annotator: {item_id: label}
    missing_value : any
        Value to use when an annotator did not label an item

    Returns
    -------
    keys : list
        Shared ordered item keys
    matrix : np.ndarray
        Shape = (num_annotators, num_items)
    """
    keys = sorted(set().union(*annotator_dicts))
    matrix = np.array([
        [ann.get(k, missing_value) for k in keys]
        for ann in annotator_dicts
    ], dtype=object)

    return keys, matrix

In [9]:
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa

def label_matrix_to_fleiss_counts(label_matrix):
    """
    Convert annotator-by-item label matrix to item-by-category count matrix.

    Parameters
    ----------
    label_matrix : np.ndarray
        Shape (n_annotators, n_items), containing categorical labels as strings.

    Returns
    -------
    count_matrix : np.ndarray
        Shape (n_items, n_categories), integer counts per item/category.
    categories : list
        Category names in column order.
    """
    label_matrix = np.asarray(label_matrix)

    if label_matrix.ndim != 2:
        raise ValueError("label_matrix must be 2D: annotators x items")

    categories = sorted(np.unique(label_matrix))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}

    n_annotators, n_items = label_matrix.shape
    count_matrix = np.zeros((n_items, len(categories)), dtype=int)

    for item_idx in range(n_items):
        for ann_idx in range(n_annotators):
            label = label_matrix[ann_idx, item_idx]
            count_matrix[item_idx, cat_to_idx[label]] += 1

    return count_matrix, categories

In [10]:
annotators = [annotated_1, annotated_2, annotated_3, annotated_4]

keys, matrix = dicts_to_annotation_matrix(annotators)

In [11]:
matrix.shape

(4, 200)

### Inter-annotation

In [15]:
def compute_fleiss_kappa_from_labels(label_matrix):
    count_matrix, categories = label_matrix_to_fleiss_counts(label_matrix)
    kappa = fleiss_kappa(count_matrix)
    return kappa, count_matrix, categories

In [16]:
kappa, count_matrix, categories = compute_fleiss_kappa_from_labels(matrix)
print(kappa)

1.0


In [17]:
import numpy as np
import krippendorff


def compute_krippendorff_alpha_from_labels(label_matrix, level_of_measurement="nominal"):
    """
    Compute Krippendorff's alpha from an annotator-by-item label matrix.

    Parameters
    ----------
    label_matrix : np.ndarray
        Shape (n_annotators, n_items), containing labels as strings or ints.
    level_of_measurement : str
        One of {"nominal", "ordinal", "interval", "ratio"}.
        For your bias labels, use "nominal".

    Returns
    -------
    alpha : float
        Krippendorff's alpha.
    encoded_matrix : np.ndarray
        Encoded numeric matrix used for computation.
    categories : list
        Category names in encoding order.
    """
    label_matrix = np.asarray(label_matrix)

    if label_matrix.ndim != 2:
        raise ValueError("label_matrix must be 2D: annotators x items")

    categories = sorted(np.unique(label_matrix))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}

    encoded_matrix = np.vectorize(cat_to_idx.get)(label_matrix).astype(float)

    alpha = krippendorff.alpha(
        reliability_data=encoded_matrix,
        level_of_measurement=level_of_measurement
    )

    return alpha, encoded_matrix, categories

In [18]:
alpha, encoded_matrix, categories = compute_krippendorff_alpha_from_labels(matrix)

print("Krippendorff's alpha:", alpha)
print("Categories:", categories)
print("Encoded matrix shape:", encoded_matrix.shape)

Krippendorff's alpha: 1.0
Categories: ['attack_discount_framing', 'attack_scarcity', 'exclusivity', 'no_bias', 'social_proof']
Encoded matrix shape: (4, 200)
